In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0,"../../")
from src.data.access import get_data, get_site_ids, aggregate_by_interval


_VIOLATION_AMOUNT = 10 # in mg/L
_WINDOW_LENGTH = "7D"

In [ ]:
def stats(site_uid, window_length = _WINDOW_LENGTH, violation_amt = _VIOLATION_AMOUNT):
    w = get_data(site_uid=site_uid).water
    start, end = w.index.min(), w.index.max()
    life = (end - start).total_seconds()/(365.25 * 3600 * 24)
    rate = w[w.nitrate_con >= violation_amt].shape[0]/w.shape[0]
    s = w['nitrate_con'].dropna()
    wk = s.resample(window_length).max()
    n = wk.notna().sum()
    return life, rate*100, n

print(f"{str("Site"):<22} {str("Lifetime (yr)"):<20} {str("Rate"):<15} {str("7-Day Windows w Data"):<20}")
for site in get_site_ids():
    print("{:<22} {:<20.2f} {:<15.1f} {:<20}".format(site, *stats(site)))

In [ ]:
# checking seasonaliity confounder
def get_mean_month_violation_number(site_uid):
    site = site_uid
    rate = '1MS'
    w = pd.DataFrame(get_data(site).water['nitrate_con'].resample('1D').max())
    w['violation'] = w.nitrate_con >= _VIOLATION_AMOUNT
    w = w.resample(rate).aggregate({'nitrate_con' : 'max', 'violation' : 'sum'}).reset_index()
    w["month"] = w["datetime"].dt.month
    avg = w[['month', 'violation']].groupby("month").agg("mean")
    return avg


import matplotlib.pyplot as plt

fig, axes = plt.subplots(2,1, figsize=(12,12))

frames = [get_mean_month_violation_number(s) for s in get_site_ids()]
sites = get_site_ids()
mean_nitrate = []
for i in range(len(frames)):
    
total = pd.concat(frames).groupby(level="month").sum()
print(total)
for frame in frames:
    axes[0].plot(frame)
axes[1].plot(total/len(get_site_ids())) 
axes[0].set_title("Avg # Violation Days Every Site")
axes[1].set_title("Avg # Days with Violation by Month Across All Sites")
fig.show()


In [ ]:
import pandas as pd

def make_markdown(df):
    lines = [[f"{col}" for col in df.columns], ["-" for col in df.columns]]
    lines += [[f"{value}" for value in df.iloc[i]] for i in range(df.shape[0])]
    lines = ["|" + "|".join(line) + "|" for line in lines]
    final = "\n".join(lines)
    return final

def keep_top_n_rows(df, n=15, score_cols=None, index_col="features"):
    """Keep rows in the top `n` of at least one score column. The `features`
    (string) column is moved to the index so it's preserved, then restored."""
    out = df.set_index(index_col) if index_col and index_col in df.columns else df

    if score_cols is None:
        num = out.select_dtypes("number")          # ranks on numeric cols only
        score_cols = [c for c in num.columns if not c.endswith("_pct")]

    keep = set()
    for c in score_cols:
        keep |= set(out[c].nlargest(n).index)

    out = out.loc[out.index.isin(keep)]
    return out.reset_index()                        # 'features' back as a column

n = 12
def print_tables(exp):
    filename = f"experiments/test_results/_experiment{exp}"
    res = pd.read_csv(filename + ".csv")
    imp1 = pd.read_csv(filename + "_importance.csv")
    imp2 = pd.read_csv(filename + "_importance_perm.csv")
    top1 = keep_top_n_rows(imp1, n=n)
    top2 = keep_top_n_rows(imp2, n=n)
    print("\n#### Results")
    print(make_markdown(res))
    print(f"\n#### Feature Importance XGBoost\n(Top {n} kept for each column)")
    print(make_markdown(top1))
    print(f"\n#### Feature Importance Col Shuffle\n(Top {n} kept for each column)")
    print(make_markdown(top2))

In [ ]:
print_tables(exp="6c")